# Stage 03 — Baseline Model (Training & Tuning)

Loads the **windowed feature dataset** from the Shared Drive, tunes the `RNNBaseline` LSTM with **Optuna (TPE) + ASHA** over **expanding-window walk-forward** cross-validation, evaluates the best configuration on a held-out test period, and summarizes the run with charts.

The same flow backs the pattern-augmented run (Stage 04) — only the input feature set changes, since one `RNNBaseline` (feature-count-agnostic) serves both.

## Environment setup (Google Colab)

Mounts the Drive, clones the repo via the `GITHUB_TOKEN` secret, and installs dependencies (including `ray[tune]` + `optuna` for tuning).

In [ ]:
import os
import sys

from google.colab import drive, userdata

drive.mount('/content/drive')

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER = 'jesseingraham'
REPO_NAME = 'comp-653-stock-prediction-i'
REPO_PATH = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_PATH):
    !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

!pip install -r requirements.txt

## Imports & configuration

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from config import DRIVE_DATA_PATH, DRIVE_MODELS_PATH
from src.models.rnn_baseline import RNNBaseline
from src.models.trainer import evaluate_on_test, tune_model
from src.utils import plotting

print('CUDA available:', torch.cuda.is_available())

## 1. Load the windowed feature dataset

> **Assumption — adjust to match the windowing stage's output.** This expects an `.npz` in the Drive `data/` folder holding:
>
> - `X` — `(n, seq_len, num_features)` float32, chronologically ordered
> - `y` — `(n, 1)` float32 next-step targets
> - `dates` — *(optional)* `(n,)` the target date of each window
>
> This is the **baseline (price-only)** feature set; Stage 04 loads the pattern-augmented file instead.

In [ ]:
# TODO: confirm filename/keys with the windowing stage.
WINDOW_FILE = 'sp500_windows_baseline.npz'

data = np.load(os.path.join(DRIVE_DATA_PATH, WINDOW_FILE), allow_pickle=True)
X = data['X'].astype('float32')
y = data['y'].astype('float32')
dates = data['dates'] if 'dates' in data.files else None

print('X:', X.shape, '| y:', y.shape, '| num_features:', X.shape[-1])

## 2. Chronological train/test split

Hold out the most recent slice as the test set; the remainder is the development set that walk-forward CV tunes on. No shuffling — order is preserved so the model never trains on the future.

In [ ]:
TEST_FRAC = 0.15  # most-recent ~15% held out for final evaluation
n_test = int(len(X) * TEST_FRAC)

X_dev, X_test = X[:-n_test], X[-n_test:]
y_dev, y_test = y[:-n_test], y[-n_test:]
test_dates = dates[-n_test:] if dates is not None else None

print(f'dev: {X_dev.shape[0]} windows | test: {X_test.shape[0]} windows')

## 3. Hyperparameter tuning (Optuna + ASHA, walk-forward)

Each trial trains one model per fold in lockstep and is scored on the mean validation RMSE across folds; ASHA stops weak trials early. Results persist to the Drive (`DRIVE_MODELS_PATH`) and are resumable after a disconnect. Flip `SMOKE_TEST` for a fast sanity run.

In [ ]:
import ray

ray.init(ignore_reinit_error=True, include_dashboard=False)

SMOKE_TEST = False
tune_kwargs = (
    dict(num_samples=4, max_epochs=6, n_folds=3)
    if SMOKE_TEST
    else dict(num_samples=30, max_epochs=50, n_folds=4)
)

results = tune_model(
    X_dev,
    y_dev,
    model_cls=RNNBaseline,
    storage_path=DRIVE_MODELS_PATH,
    experiment_name='baseline_rnn_tune',
    gpu_per_trial=0.25 if torch.cuda.is_available() else 0.0,
    **tune_kwargs,
)

best = results.get_best_result('val_rmse', mode='min')
print('Best config:', best.config)
print('Best mean val RMSE:', round(best.metrics['val_rmse'], 5))

## 4. Tuning summary & charts

The 10 best configurations, then: per-trial performance, learning curves across trials (short curves = ASHA-stopped), and how validation RMSE responds to each hyperparameter.

In [ ]:
results_df = results.get_dataframe()
results_df.sort_values('val_rmse').head(10)

In [ ]:
plotting.plot_trial_performance(results_df, metric='val_rmse', mode='min')
plt.show()

In [ ]:
curves = [r.metrics_dataframe for r in results]
final_vals = [
    c['val_rmse'].iloc[-1] if 'val_rmse' in c else np.inf for c in curves
]
best_index = int(np.argmin(final_vals))

plotting.plot_learning_curves(curves, metric='val_rmse', best_index=best_index)
plt.show()

In [ ]:
params = [
    f'config/{p}'
    for p in ['lr', 'hidden_size', 'num_layers', 'dropout',
              'weight_decay', 'batch_size']
]
plotting.plot_hyperparameter_effects(results_df, params, metric='val_rmse')
plt.show()

## 5. Final evaluation on the held-out test set

Retrain the best configuration on **all** development data, then score the untouched test period — reporting RMSE and directional (up/down) accuracy.

In [ ]:
res = evaluate_on_test(
    best.config,
    X_dev,
    y_dev,
    X_test,
    y_test,
    model_cls=RNNBaseline,
    max_epochs=tune_kwargs['max_epochs'],
)

print(f"Test RMSE:                 {res['test_rmse']:.5f}")
print(f"Test directional accuracy: {res['test_directional_accuracy']:.3f}")

In [ ]:
plotting.plot_predictions_vs_actual(
    res['targets'], res['predictions'], index=test_dates,
    title='Baseline RNN — test set: predicted vs. actual',
)
plt.show()

## Summary

The baseline `RNNBaseline` has been tuned via walk-forward CV and evaluated on the held-out test period, with the tuning run and test predictions visualized above. **Next:** Stage 04 repeats this exact flow on the pattern-augmented windowed dataset, so the test RMSE / directional accuracy can be compared head-to-head to measure whether pattern features help.